In [1]:
!pip install -q kagglehub pandas
!pip install spacy
!python -m spacy download en_core_web_sm

import kagglehub
import pandas as pd
import os
import ast
import spacy
import numpy as np
import os
import shutil
from sentence_transformers import SentenceTransformer
from tqdm import tqdm



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 112.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
news = pd.read_csv('/content/drive/MyDrive/data/news/news_processed.csv')

# News Embeddings

Features that will be included into the news Embedding include:
- title
- abstract
- category
- title_entities
- abstract_entities





In [4]:
! pip install sentence_transformers
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cpu')

news_with_embeddings = news.copy()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## Title Embeddings

In [5]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Ensure model is on GPU if available
device = 'cuda' if model.device.type == 'cuda' else 'cpu'

# Prepare text input
titles = news_with_embeddings['title'].fillna("").tolist()  # small list, ok

# Generate embeddings efficiently
title_vecs = model.encode(
    titles,
    batch_size=128,
    show_progress_bar=True,
    device=device,
    convert_to_numpy=True,
)

# L2 normalize and convert to float32
title_vecs = title_vecs.astype(np.float32)
title_vecs /= np.linalg.norm(title_vecs, axis=1, keepdims=True)

# Store embeddings in a separate array for efficiency
news_title_embeddings = title_vecs  # shape: (num_rows, embedding_dim)

# store references in the DataFrame for indexing/search
news_with_embeddings['title_embedding_idx'] = np.arange(len(news_with_embeddings))

Batches:   0%|          | 0/401 [00:00<?, ?it/s]

In [6]:
#Check if the embeddings is really stored within news_title_embeddings
idx = news_with_embeddings.loc[0, 'title_embedding_idx']
embedding = news_title_embeddings[idx]

print(embedding.shape)  # e.g., (384,)
print(embedding[:10])

(384,)
[-0.00928648  0.04244234  0.05898828  0.01210564  0.03341918  0.01964553
  0.02736862 -0.06152726 -0.0361674  -0.0390474 ]


## Abstract Embeddings

In [7]:
import numpy as np
from tqdm import tqdm  # for a single progress bar

# Prepare text input
abstracts = news_with_embeddings['abstract'].fillna(news_with_embeddings['title']).tolist()
chunk_size = 500  # adjust based on RAM/GPU

all_vecs = []

# Use tqdm for a single progress bar
for start in tqdm(range(0, len(abstracts), chunk_size), desc="Encoding abstracts"):
    batch = abstracts[start:start + chunk_size]

    # Encode without internal progress bar
    batch_vecs = model.encode(
        batch,
        batch_size=128,
        device=device,
        convert_to_numpy=True,
        show_progress_bar=False
    )

    # L2 normalize and convert to float32
    batch_vecs = batch_vecs.astype(np.float32)
    batch_vecs /= np.linalg.norm(batch_vecs, axis=1, keepdims=True)

    all_vecs.append(batch_vecs)

# Combine all chunks into a single NumPy array
news_abstract_embeddings = np.vstack(all_vecs)

# Add reference index to DataFrame for easy lookup
news_with_embeddings['abstract_embedding_idx'] = np.arange(len(news_with_embeddings))



Encoding abstracts: 100%|██████████| 103/103 [28:47<00:00, 16.77s/it]


In [8]:
#Check if the embeddings is really stored within news_title_embeddings
idx = news_with_embeddings.loc[0, 'abstract_embedding_idx']
embedding = news_abstract_embeddings[idx]

print(embedding.shape)
print(embedding[:10])

(384,)
[-0.01846291  0.09640496  0.06174493 -0.03290942  0.07545112  0.03379141
 -0.00471013  0.00211287 -0.00625122  0.03798711]


In [9]:
import numpy as np

# Combine title and abstract embeddings
title_abstract_embeddings = np.hstack([
    news_title_embeddings[news_with_embeddings['title_embedding_idx'].to_numpy()],
    news_abstract_embeddings[news_with_embeddings['abstract_embedding_idx'].to_numpy()]
]).astype(np.float32)

# Normalize the new combined embeddings
title_abstract_embeddings /= np.linalg.norm(title_abstract_embeddings, axis=1, keepdims=True)

print("Shape of combined title + abstract embeddings:", title_abstract_embeddings.shape)
print("First 10 dimensions of first article (title + abstract):", title_abstract_embeddings[0][:10])

# Save the new embeddings locally
np.save("title_abstract_embeddings.npy", title_abstract_embeddings)
print("Saved title_abstract_embeddings.npy locally.")

Shape of combined title + abstract embeddings: (51282, 768)
First 10 dimensions of first article (title + abstract): [-0.00656653  0.03001126  0.041711    0.00855998  0.02363093  0.01389148
  0.01935253 -0.04350633 -0.02557421 -0.02761068]
Saved title_abstract_embeddings.npy locally.


In [18]:
import shutil
import os
drive_path_ta = '/content/drive/MyDrive/embeddings/title_abstract_embeddings.npy'

source_path_ta = 'title_abstract_embeddings.npy'

if os.path.exists(source_path_ta):
    shutil.copy(source_path_ta, drive_path_ta)
    print(f'File saved to {drive_path_ta}')
else:
    print(f'Error: {source_path_ta} not found.')

File saved to /content/drive/MyDrive/embeddings/title_abstract_embeddings.npy
